# DATA PREPROCESSING & MODEL TRAINING
## Proyek Sistem Cerdas: Prediksi Harga Mobil Bekas di Indonesia

Notebook ini memuat proses preprocessing data, pembersihan outlier, pembangunan arsitektur Jaringan Saraf Tiruan (ANN) tipe MLP menggunakan Keras/TensorFlow, training dengan target log-transformed untuk optimasi MAPE, dan evaluasi performa model regresi harga mobil bekas.

### 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os

sns.set_theme(style='whitegrid')
print('TensorFlow version:', tf.__version__)

ModuleNotFoundError: No module named 'seaborn'

### 2. Memuat dan Eksplorasi Dataset

In [ ]:
df = pd.read_csv('mobil_bekas_carmudi.csv')
print('Jumlah Data Awal:', len(df))
df.head()

### 3. Pembersihan Outlier & Pemilihan Brand Populer

In [ ]:
popular_brands = ['Toyota', 'Honda', 'Suzuki', 'Mitsubishi', 'Daihatsu', 'Hyundai', 'Wuling', 'Nissan']
df = df[df['brand'].isin(popular_brands)]

df = df[(df['price'] >= 60_000_000) & (df['price'] <= 800_000_000)]
print('Jumlah Data Setelah Filter:', len(df))

### 4. Preprocessing Data & Feature Engineering

In [ ]:
def clean_model_name(model_str):
    words = str(model_str).strip().split()
    if words:
        val = words[0]
        return ''.join(c for c in val if c.isalnum())
    return 'Unknown'

def clean_location(loc_str):
    loc_lower = str(loc_str).lower()
    if 'jakarta' in loc_lower: return 'DKI Jakarta'
    elif 'banten' in loc_lower or 'tangerang' in loc_lower: return 'Banten'
    elif 'jawa barat' in loc_lower or 'bandung' in loc_lower: return 'Jawa Barat'
    elif 'jawa tengah' in loc_lower: return 'Jawa Tengah'
    elif 'jawa timur' in loc_lower: return 'Jawa Timur'
    elif 'yogyakarta' in loc_lower: return 'Yogyakarta'
    elif 'bali' in loc_lower: return 'Bali'
    elif 'sumatera' in loc_lower: return 'Sumatera'
    elif 'kalimantan' in loc_lower: return 'Kalimantan'
    elif 'sulawesi' in loc_lower: return 'Sulawesi'
    else: return 'Lainnya'

df['age'] = 2026 - df['year']
df['model_clean'] = df['model'].apply(clean_model_name)
df['location_clean'] = df['location'].apply(clean_location)

model_counts = df['model_clean'].value_counts()
top_models = model_counts[model_counts >= 3].index.tolist()
df['model_clean'] = df['model_clean'].apply(lambda x: x if x in top_models else 'Lainnya')

df.head()

### 5. Encoding & Scaling

In [ ]:
cat_cols = ['brand', 'model_clean', 'transmission', 'location_clean', 'fuel_type']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False)

dummy_columns = [col for col in df_encoded.columns if any(col.startswith(cat + '_') for cat in cat_cols)]

num_cols = ['age', 'mileage']
scaler = StandardScaler()

X_num = scaler.fit_transform(df_encoded[num_cols])
X_cat = df_encoded[dummy_columns].values.astype(np.float32)
X = np.hstack([X_num, X_cat])

# Target log Juta Rupiah
y = np.log(df['price'].values / 1e6).astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
print('Dimensi Input:', X.shape[1])

### 6. Membangun Jaringan Saraf Tiruan (ANN)

In [ ]:
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='linear')
])

optimizer = keras.optimizers.Adam(learning_rate=0.005)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
model.summary()

### 7. Melatih Model

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=150,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

### 8. Evaluasi Performa Model (Original Scale)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss (MSE)')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)')
plt.title('Kurva Belajar Model')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.legend()
plt.show()

y_pred_log = model.predict(X_test).flatten()
y_test_orig = np.exp(y_test)
y_pred_orig = np.exp(y_pred_log)

mae = np.mean(np.abs(y_test_orig - y_pred_orig))
mape = np.mean(np.abs((y_test_orig - y_pred_orig) / y_test_orig)) * 100
r2 = 1 - (np.sum((y_test_orig - y_pred_orig) ** 2) / np.sum((y_test_orig - np.mean(y_test_orig)) ** 2))

print(f'Test MAE: {mae:.2f} Juta Rupiah')
print(f'Test MAPE: {mape:.2f}%')
print(f'R2 Score: {r2:.4f}')

### 9. Menyimpan Model

In [ ]:
model.save('model_harga_mobil.h5')
print('Model berhasil disimpan ke model_harga_mobil.h5')